In [2]:
import numpy as np
import time
import psutil
import os

def monitor_memory():
    """Monitor memory usage"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # MB

print("Demonstrating the Computational Challenge of Normal Equation for High-Dimensional Data")
print("=" * 70)

# Parameters for the simulation
n_samples = 1000
n_features = 1000  # Reduced for feasibility but still demonstrates the issue

print(f"Number of samples: {n_samples}")
print(f"Number of features: {n_features}")
print(f"Shape of matrix X: ({n_samples}, {n_features})")
print(f"Shape of matrix X^T X: ({n_features}, {n_features})")

# Generate synthetic data
print("\n1. Generating synthetic data...")
start_time = time.time()
np.random.seed(42)
X = np.random.randn(n_samples, n_features)
true_theta = np.random.randn(n_features)
y = X @ true_theta + 0.1 * np.random.randn(n_samples)  # y = Xθ + noise

print(f"Data generation time: {time.time() - start_time:.2f} seconds")
print(f"Memory usage after data generation: {monitor_memory():.2f} MB")

# Normal Equation Method
print("\n2. Solving using Normal Equation Method...")
print("   Step 1: Computing X^T X...")
start_time = time.time()

try:
    # Compute X^T X
    memory_before = monitor_memory()
    X_transpose_X = X.T @ X
    memory_after = monitor_memory()
    
    print(f"   Time for X^T X computation: {time.time() - start_time:.2f} seconds")
    print(f"   Memory used for X^T X: {memory_after - memory_before:.2f} MB")
    print(f"   Shape of X^T X matrix: {X_transpose_X.shape}")
    
    # Check matrix properties
    print("   Step 2: Checking matrix properties...")
    cond_number = np.linalg.cond(X_transpose_X)
    rank = np.linalg.matrix_rank(X_transpose_X)
    print(f"   Condition number: {cond_number:.2e}")
    print(f"   Matrix rank: {rank}/{n_features}")
    
    if cond_number > 1e15:
        print("   ⚠️  Matrix is ill-conditioned! Inverse computation may be unstable.")
    
    # Compute inverse - This is the computationally expensive step!
    print("   Step 3: Computing matrix inverse (X^T X)^-1...")
    start_inv = time.time()
    
    try:
        X_transpose_X_inv = np.linalg.inv(X_transpose_X)
        inv_time = time.time() - start_inv
        print(f"   ✅ Matrix inverse computed successfully")
        print(f"   Time for inverse computation: {inv_time:.2f} seconds")
        print(f"   Shape of inverse matrix: {X_transpose_X_inv.shape}")
        
        # Verify inverse correctness
        print("   Step 4: Verifying inverse correctness...")
        identity_approx = X_transpose_X @ X_transpose_X_inv
        identity_error = np.linalg.norm(identity_approx - np.eye(n_features))
        print(f"   Inverse error ||(X^T X)(X^T X)^-1 - I||: {identity_error:.2e}")
        
        # Compute parameters θ = (X^T X)^-1 X^T y
        print("   Step 5: Computing parameters θ = (X^T X)^-1 X^T y...")
        theta_normal = X_transpose_X_inv @ (X.T @ y)
        print(f"   ✅ Parameters computed successfully")
        
        # Calculate error
        theta_error = np.linalg.norm(theta_normal - true_theta)
        print(f"   Parameter error ||θ_normal - θ_true||: {theta_error:.4f}")
        
    except np.linalg.LinAlgError as e:
        print(f"   ❌ Error in matrix inversion: {e}")
        
except MemoryError:
    print("   ❌ Memory error! Not enough RAM.")
except Exception as e:
    print(f"   ❌ Unexpected error: {e}")

total_normal_time = time.time() - start_time
print(f"Total time for Normal Equation method: {total_normal_time:.2f} seconds")

# Compare with sklearn's implementation (uses SVD/LSQR internally)
print("\n3. Solving using sklearn's LinearRegression (uses SVD)...")
start_time = time.time()

try:
    from sklearn.linear_model import LinearRegression
    
    model = LinearRegression(fit_intercept=False)
    model.fit(X, y)
    theta_sklearn = model.coef_
    
    print(f"   ✅ Model trained successfully")
    print(f"   Training time: {time.time() - start_time:.2f} seconds")
    
    # Calculate error
    theta_error_sklearn = np.linalg.norm(theta_sklearn - true_theta)
    print(f"   Parameter error ||θ_sklearn - θ_true||: {theta_error_sklearn:.4f}")
    print(f"   R² Score: {model.score(X, y):.6f}")
    
except Exception as e:
    print(f"   ❌ Error in sklearn method: {e}")

# Computational complexity analysis
print("\n" + "="*70)
print("COMPUTATIONAL COMPLEXITY ANALYSIS")
print("="*70)

# Calculate number of operations
matrix_multiplication_ops = n_samples * n_features * n_features  # X^T X
matrix_inversion_ops = n_features ** 3  # Inverse computation
total_ops_normal = matrix_multiplication_ops + matrix_inversion_ops

print(f"Matrix multiplication (X^T X) operations: {matrix_multiplication_ops:,}")
print(f"Matrix inversion operations: ~{matrix_inversion_ops:,}")
print(f"Total operations for Normal Equation: ~{total_ops_normal:,}")

# Estimate time based on operations (assuming 1e9 operations per second)
estimated_time = total_ops_normal / 1e9
print(f"Estimated computation time: {estimated_time:.2f} seconds")

# Show how complexity grows with dimension
print("\nGROWTH OF COMPUTATIONAL COMPLEXITY (O(d³)):")
print("Dimensions | Operations  | Estimated Time")
print("-" * 45)

dimensions = [100, 500, 1000, 2000, 5000, 10000]
for d in dimensions:
    ops = d ** 3  # Dominant term is O(d³)
    time_est = ops / 1e9  # Assuming 1e9 operations per second
    
    if d <= n_features:
        status = "(feasible)" if d <= 1000 else "(challenging)"
    else:
        status = "(very slow)" if d <= 5000 else "(impractical)"
    
    print(f"d = {d:5} | {ops:12,} | {time_est:6.1f} sec {status}")

print("\n" + "="*70)
print("KEY INSIGHTS:")
print("="*70)
print("""
1. MEMORY BOTTLENECK:
   - X^T X matrix for d=1000 requires storing 1,000,000 elements
   - For d=10,000, it's 100,000,000 elements (~800 MB for float64)

2. COMPUTATIONAL COMPLEXITY:
   - Matrix inversion is O(d³) - grows cubically with dimensions
   - For d=1000: ~1e9 operations
   - For d=5000: ~125e9 operations (125x more!)

3. NUMERICAL STABILITY:
   - X^T X often becomes ill-conditioned for high dimensions
   - Small errors in computation get amplified

4. PRACTICAL SOLUTIONS:
   - Use iterative methods (Gradient Descent)
   - Use matrix factorization (SVD, QR)
   - Use regularization (Ridge Regression)
   - Use stochastic methods for very large datasets
""")

# Additional: Compare with pseudo-inverse (more stable)
print("\n4. Alternative: Using Moore-Penrose Pseudo-inverse...")
start_time = time.time()

try:
    X_pseudo_inv = np.linalg.pinv(X)  # More numerically stable
    theta_pseudo = X_pseudo_inv @ y
    
    print(f"   ✅ Pseudo-inverse computed successfully")
    print(f"   Computation time: {time.time() - start_time:.2f} seconds")
    
    theta_error_pseudo = np.linalg.norm(theta_pseudo - true_theta)
    print(f"   Parameter error ||θ_pseudo - θ_true||: {theta_error_pseudo:.4f}")
    
except Exception as e:
    print(f"   ❌ Error in pseudo-inverse computation: {e}")

print("\n" + "="*70)
print("CONCLUSION: Normal Equation becomes impractical for d > 1000")
print("Use iterative methods like Gradient Descent for high-dimensional data!")
print("="*70)

Demonstrating the Computational Challenge of Normal Equation for High-Dimensional Data
Number of samples: 1000
Number of features: 1000
Shape of matrix X: (1000, 1000)
Shape of matrix X^T X: (1000, 1000)

1. Generating synthetic data...
Data generation time: 0.03 seconds
Memory usage after data generation: 557.11 MB

2. Solving using Normal Equation Method...
   Step 1: Computing X^T X...
   Time for X^T X computation: 0.04 seconds
   Memory used for X^T X: -183.11 MB
   Shape of X^T X matrix: (1000, 1000)
   Step 2: Checking matrix properties...
   Condition number: 8.07e+07
   Matrix rank: 1000/1000
   Step 3: Computing matrix inverse (X^T X)^-1...
   ✅ Matrix inverse computed successfully
   Time for inverse computation: 0.09 seconds
   Shape of inverse matrix: (1000, 1000)
   Step 4: Verifying inverse correctness...
   Inverse error ||(X^T X)(X^T X)^-1 - I||: 1.31e-08
   Step 5: Computing parameters θ = (X^T X)^-1 X^T y...
   ✅ Parameters computed successfully
   Parameter error ||